In [ ]:
import os
import sys
import time
import gc
from typing import List, Tuple
import pickle
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
project_path = os.path.abspath(os.path.dirname(os.getcwd()))
sys.path.append(project_path)
from src import *

In [ ]:
experiment =  'stella_maris_pretrain_56channel_all_windows_final'
experiment_path = os.path.join(project_path, 'results', experiment)
spike_path = os.path.join(experiment_path, 'spikes')
spike_list = os.listdir(spike_path)

In [ ]:
all_windows = []
subject_labels = {}

for spike_file in spike_list:
    if not spike_file.endswith(".npz"):
        continue

    subj = spike_file.replace("_spikes.npz","")
    spikes = np.load(os.path.join(spike_path, spike_file))

    all_spikes = spikes["concat_raw"]
    labels_raw = spikes["labels_raw"]
    window_lengths = spikes["window_lengths"]
    if 'SC' in spike_file and int(labels_raw[0]) == 1:
        print(spike_file)
        continue
    subject_labels[subj] = int(labels_raw[0])
    ptr = 0
    for L in window_lengths:
        w = all_spikes[ptr:ptr+L,:]
        y = int(labels_raw[ptr])
        all_windows.append((w,y,subj))
        ptr += L

In [ ]:
subject_list = list(subject_labels.keys())
subject_y = np.array([subject_labels[s] for s in subject_list])

In [ ]:
subject_list

In [ ]:
def extract_subject_features_from_spikes(subject_spikes_paths):
    """
    Ricostruisce subject_features esattamente come in pretrain.py
    usando SOLO i file .npz (filter_raw, source_raw, window_lengths).

    Output:
        subject_features_new : dict[subj] = feature_vector (2N,)
    """

    subject_features_new = {}

    for subj, path in subject_spikes_paths.items():

        data = np.load(path)
        filter_raw = data["filter_raw"]          # (T_tot, N)
        source_raw = data["source_raw"]          # (T_tot, N)
        window_lengths = data["window_lengths"]  # (W,)
        labels_raw = data["labels_raw"]          # (T_tot,)  # non lo usiamo

        # ricostruzione identica al pretrain
        ptr = 0
        window_feats = []

        for L in window_lengths:
            f_mean = filter_raw[ptr:ptr+L].mean(axis=0)    # (N,)
            s_mean = source_raw[ptr:ptr+L].mean(axis=0)    # (N,)
            feat_window = np.concatenate([f_mean, s_mean]) # (2N,)
            window_feats.append(feat_window)
            ptr += L

        # media su tutte le finestre del soggetto
        subject_features_new[subj] = np.mean(window_feats, axis=0)

    return subject_features_new

def compute_global_report(confusion_matrices):
    cm = np.sum(confusion_matrices, axis=0)
    tn, fp, fn, tp = cm.ravel()

    precision_pos = tp / (tp + fp) if tp + fp > 0 else 0
    recall_pos    = tp / (tp + fn) if tp + fn > 0 else 0
    f1_pos = 2 * precision_pos * recall_pos / (precision_pos + recall_pos) if precision_pos + recall_pos > 0 else 0

    precision_neg = tn / (tn + fn) if tn + fn > 0 else 0
    recall_neg    = tn / (tn + fp) if tn + fp > 0 else 0
    f1_neg = 2 * precision_neg * recall_neg / (precision_neg + recall_neg) if precision_neg + recall_neg > 0 else 0

    accuracy = (tp + tn) / (tp + tn + fp + fn)

    report = {
        "accuracy": accuracy,
        "class_0": {
            "precision": precision_neg,
            "recall": recall_neg,
            "f1-score": f1_neg
        },
        "class_1": {
            "precision": precision_pos,
            "recall": recall_pos,
            "f1-score": f1_pos
        }
    }

    return cm, report


In [ ]:
import os
import pickle

load_path = os.path.join(experiment_path, "subject_features.pkl")
if not os.path.exists(load_path):
    load_path = os.path.join(project_path, 'results', experiment, "subject_features.pkl")

with open(load_path, "rb") as f:
    data_loaded = pickle.load(f)

subject_features = data_loaded["features"]
subject_labels = data_loaded["labels"]
subjects_list  = data_loaded["subjects"]

spike_dir = spike_path   # la tua directory, già definita
subject_spikes_paths = {}
subject_features_dict = {}
for idx, fname in enumerate(spike_list):
    if fname.endswith("_spikes.npz"):
        subj = fname.replace("_spikes.npz", "")
        if subj in ['SC_037', 'SC_068', 'SC_054', 'SC_72', 'SC_86']:
            continue
        subject_spikes_paths[subj] = os.path.join(spike_dir, fname)
     #   subject_features_dict[subj] = subject_features[idx]

# controllo
print("Num subjects:", len(subject_spikes_paths))
print("Esempio:", list(subject_spikes_paths.items())[:5])

subject_features_from_spikes = extract_subject_features_from_spikes(subject_spikes_paths)


In [ ]:
# ===== REBUILD subject_to_spike_windows FROM NPZ =====

subject_to_spike_windows = {}
subject_labels_npz = {}

for subj, path in subject_spikes_paths.items():

    data = np.load(path)
    concat_raw = data["concat_raw"]            # (T_tot, 1008)
    window_lengths = data["window_lengths"]    # array (num_windows,)
    labels = data["labels_raw"]                # (T_tot,)

    subject_labels_npz[subj] = int(labels[0])  # unica label del soggetto

    ptr = 0
    windows = []

    for L in window_lengths:
        w = concat_raw[ptr:ptr+L, :]           # estrai finestra (L × 1008)
        windows.append(w)
        ptr += L

    subject_to_spike_windows[subj] = windows

print("OK ✓ Ricostruite tutte le finestre dai .npz")
print("Numero soggetti:", len(subject_to_spike_windows))
print("Esempio finestre soggetto:", list(subject_to_spike_windows.keys())[0],
      "num_finestra =", len(subject_to_spike_windows[list(subject_to_spike_windows.keys())[0]]))
print("windows dimensioni: ", w.shape)


In [ ]:
import numpy as np
from scipy.stats import entropy

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns


# ============================================================
# Parameters (match paper)
# ============================================================
PCA_VAR_THRESHOLD = 0.85     # retain PCs explaining >=85% variance
N_SPLITS = 5                 # stratified 5-fold CV at subject level

TB_BINS = 20                 # Temporal Binning (20 bins) -> 2 s / 20 = 100 ms per bin
THP_BINS = 9                 # Temporal Histogram Pooling (9 bins) on normalized time
BI_SMOOTH_BINS = 20          # smoothing bins for Burstiness Index (peak-to-peak)

SVM_C = 1.0
SVM_GAMMA = "scale"
SVM_KERNEL = "rbf"
SVM_CLASS_WEIGHT = "balanced"
RANDOM_STATE = 42


# ============================================================
# Feature functions (window-level), then subject-level averaging
# sp is expected as (T, N) binary spikes
# ============================================================
def afr_window(sp):
    """AFR: Average Firing Rate per neuron: mean over time. Returns (N,)."""
    return np.mean(sp, axis=0)

def tb_window(sp, bins=TB_BINS):
    """
    TB: Temporal Binning.
    Partition time into 'bins' segments and compute mean firing rate per bin.
    Returns (N*bins,).
    """
    T, N = sp.shape
    edges = np.linspace(0, T, bins + 1, dtype=int)

    feat = np.zeros((N, bins), dtype=float)
    for b in range(bins):
        s, e = edges[b], edges[b + 1]
        feat[:, b] = np.mean(sp[s:e, :], axis=0) if e > s else 0.0

    return feat.flatten()

def thp_window(sp, bins=THP_BINS):
    """
    THP: Temporal Histogram Pooling.
    Histogram of spike occurrence times on normalized time [0,1], per neuron.
    Returns (N*bins,).
    """
    T, N = sp.shape
    time_norm = np.linspace(0, 1, T, endpoint=False)
    edges = np.linspace(0, 1, bins + 1)

    feat = np.zeros((N, bins), dtype=float)
    for i in range(N):
        spike_t = time_norm[sp[:, i] > 0]
        h, _ = np.histogram(spike_t, bins=edges)
        feat[i, :] = h

    return feat.flatten()

def bi_window(sp, smooth_bins=BI_SMOOTH_BINS):
    """
    BI: Burstiness Index (peak-to-peak).
    Compute smoothed firing rate profile (bins) and take max-min per neuron.
    Returns (N,).
    """
    T, N = sp.shape
    edges = np.linspace(0, T, smooth_bins + 1, dtype=int)

    fr = np.zeros((smooth_bins, N), dtype=float)
    for b in range(smooth_bins):
        s, e = edges[b], edges[b + 1]
        fr[b, :] = np.mean(sp[s:e, :], axis=0) if e > s else 0.0

    return np.max(fr, axis=0) - np.min(fr, axis=0)

def te_window(sp, bins=TB_BINS):
    """
    TE: Temporal Entropy (optional).
    Entropy of spike count distribution across time bins, per neuron.
    Returns (N,).
    """
    T, N = sp.shape
    edges = np.linspace(0, T, bins + 1, dtype=int)
    feat = np.zeros(N, dtype=float)

    for i in range(N):
        counts = np.zeros(bins, dtype=float)
        for b in range(bins):
            s, e = edges[b], edges[b + 1]
            counts[b] = np.sum(sp[s:e, i]) if e > s else 0.0

        tot = np.sum(counts)
        if tot <= 0:
            feat[i] = 0.0
        else:
            p = counts / tot
            feat[i] = entropy(p)

    return feat

def concat_features(*vecs):
    """Utility for AFR+BI, etc."""
    return np.concatenate(vecs, axis=0)

def extract_subject_features(subject_to_spike_windows, feature_fn):
    """
    subject_to_spike_windows: dict[subj] -> list of (T_i, N)
    feature_fn: maps (T,N) -> (D,)
    Returns:
        X: (n_subjects, D)
        subjects: list of subject IDs
    """
    X, subjects = [], []
    for subj in sorted(subject_to_spike_windows.keys()):
        windows = subject_to_spike_windows[subj]
        feats = [feature_fn(w) for w in windows]
        feats = np.stack(feats, axis=0)          # (n_windows, D)
        X.append(np.mean(feats, axis=0))         # subject-level average
        subjects.append(subj)
    return np.asarray(X), subjects


In [ ]:
def metrics_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()
    tot = tn + fp + fn + tp

    acc = (tp + tn) / tot if tot else np.nan
    sens = tp / (tp + fn) if (tp + fn) else np.nan          # TPR (class 1 recall)
    spec = tn / (tn + fp) if (tn + fp) else np.nan          # TNR (class 0 recall)
    bacc = (sens + spec) / 2 if np.isfinite(sens) and np.isfinite(spec) else np.nan

    return acc, bacc, spec, sens


In [ ]:


# ============================================================
# Labels (subject-level)
# ============================================================
subjects = sorted(subject_to_spike_windows.keys())
y = np.array([subject_labels_npz[s] for s in subjects])


# ============================================================
# Feature representations (match paper naming)
# ============================================================
feature_methods = [
    ("AFR", lambda sp: afr_window(sp)),
    ("TB (20 bins)", lambda sp: tb_window(sp, bins=TB_BINS)),
    ("THP (9 bins)", lambda sp: thp_window(sp, bins=THP_BINS)),
    ("BI (peak-to-peak)", lambda sp: bi_window(sp, smooth_bins=BI_SMOOTH_BINS)),
    ("AFR+BI", lambda sp: concat_features(afr_window(sp), bi_window(sp, smooth_bins=BI_SMOOTH_BINS))),
    # ("TE", lambda sp: te_window(sp, bins=TB_BINS)),  # optional
]


# ============================================================
# CV + model pipeline
# ============================================================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=PCA_VAR_THRESHOLD, svd_solver="full")),
    ("svm", SVC(
        kernel=SVM_KERNEL,
        probability=True,
        class_weight=SVM_CLASS_WEIGHT,
        gamma=SVM_GAMMA,
        C=SVM_C
    ))
])

for feat_name, feat_fn in feature_methods:

    X_raw, _ = extract_subject_features(subject_to_spike_windows, feature_fn=feat_fn)

    conf_mats = []
    fold_metrics = []

    for tr, ts in skf.split(X_raw, y):
        model.fit(X_raw[tr], y[tr])
        y_pred = model.predict(X_raw[ts])

        cm = confusion_matrix(y[ts], y_pred, labels=[0, 1])
        conf_mats.append(cm)
        fold_metrics.append(metrics_from_cm(cm))

    cm_tot = np.sum(conf_mats, axis=0)

    acc, bacc, spec, sens = metrics_from_cm(cm_tot)

    fold_metrics = np.array(fold_metrics, dtype=float)
    m = np.nanmean(fold_metrics, axis=0)
    s = np.nanstd(fold_metrics, axis=0)

    print("\n" + "=" * 60)
    print(f"FEATURE SET: {feat_name}")
    print("=" * 60)
    print("Aggregated CM metrics:")
    print(f"  Accuracy          : {acc:.4f}")
    print(f"  Balanced Accuracy : {bacc:.4f}")
    print(f"  Specificity (TNR) : {spec:.4f}")
    print(f"  Sensitivity (TPR) : {sens:.4f}")

    print("\nAcross-fold metrics (mean ± std):")
    print(f"  Accuracy          : {m[0]:.4f} ± {s[0]:.4f}")
    print(f"  Balanced Accuracy : {m[1]:.4f} ± {s[1]:.4f}")
    print(f"  Specificity (TNR) : {m[2]:.4f} ± {s[2]:.4f}")
    print(f"  Sensitivity (TPR) : {m[3]:.4f} ± {s[3]:.4f}")

    # Confusion matrix plot (aggregated)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm_tot,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["True 0", "True 1"]
    )
    plt.title(f"Confusion Matrix – {feat_name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
from scipy.stats import entropy

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns


# ============================================================
# Parameters (match paper)
# ============================================================
PCA_VAR_THRESHOLD = 0.85     # retain PCs explaining >=85% variance
PCA_VAR_THRESHOLD = 0.5     # retain PCs explaining >=85% variance
N_SPLITS = 5                 # stratified 5-fold CV at subject level

# Your window is T=2s. With 20 bins -> 100 ms/bin (coarse but robust)
TB_BINS = 20                 # Temporal Binning bins
BI_SMOOTH_BINS = 20          # smoothing bins used for Burstiness peak-to-peak

# Reservoir topology
N_LAYERS = 56
N_PER_LAYER = 9              # 3x3 neurons per layer
N_NEURONS = N_LAYERS * N_PER_LAYER  # 1008

# SVM hyperparameters (kept fixed across feature sets)
SVM_C = 1.0
SVM_GAMMA = "scale"
SVM_KERNEL = "rbf"
SVM_CLASS_WEIGHT = "balanced"
RANDOM_STATE = 42

N_LAYERS = 56
N_PER_LAYER = 9
N_ONE_RES = N_LAYERS * N_PER_LAYER  # 504
N_TOTAL = 2 * N_ONE_RES             # 1008 (source+filter)

def split_source_filter(sp):
    T, N = sp.shape
    assert N == N_TOTAL, f"Expected {N_TOTAL} (source+filter), got {N}"
    sp_src = sp[:, :N_ONE_RES]
    sp_flt = sp[:, N_ONE_RES:]
    return sp_src, sp_flt

def to_layer_tensor_one_res(sp_res):
    T, N = sp_res.shape
    assert N == N_ONE_RES, f"Expected {N_ONE_RES} neurons, got {N}"
    return sp_res.reshape(T, N_LAYERS, N_PER_LAYER)

def sp_layer_mean_dual(sp):
    """
    Spatial pooling within each layer (3x3) for BOTH reservoirs, then concat.
    sp: (T,1008) -> returns (T, 56*2) = (T,112)
    """
    sp_src, sp_flt = split_source_filter(sp)

    S_src = to_layer_tensor_one_res(sp_src)  # (T,56,9)
    S_flt = to_layer_tensor_one_res(sp_flt)  # (T,56,9)

    L_src = np.mean(S_src, axis=2)           # (T,56)
    L_flt = np.mean(S_flt, axis=2)           # (T,56)

    return np.concatenate([L_src, L_flt], axis=1)  # (T,112)

def lafr_window(sp):
    """L-AFR (dual): mean over time of layer-pooled activity. -> (112,)"""
    L = sp_layer_mean_dual(sp)               # (T,112)
    return np.mean(L, axis=0)

def ltb_window(sp, bins=TB_BINS):
    """L-TB (dual): temporal binning on layer-pooled activity. -> (112*bins,)"""
    L = sp_layer_mean_dual(sp)               # (T,112)
    T, D = L.shape
    edges = np.linspace(0, T, bins + 1, dtype=int)

    feat = np.zeros((D, bins), dtype=float)
    for b in range(bins):
        s, e = edges[b], edges[b+1]
        feat[:, b] = np.mean(L[s:e, :], axis=0) if e > s else 0.0

    return feat.flatten()

def lbi_window(sp, smooth_bins=BI_SMOOTH_BINS):
    """L-BI (dual): peak-to-peak of smoothed layer activity. -> (112,)"""
    L = sp_layer_mean_dual(sp)               # (T,112)
    T, D = L.shape
    edges = np.linspace(0, T, smooth_bins + 1, dtype=int)

    fr = np.zeros((smooth_bins, D), dtype=float)
    for b in range(smooth_bins):
        s, e = edges[b], edges[b+1]
        fr[b, :] = np.mean(L[s:e, :], axis=0) if e > s else 0.0

    return np.max(fr, axis=0) - np.min(fr, axis=0)



# ============================================================
# Subject-level aggregation
# ============================================================
def extract_subject_features(subject_to_spike_windows, feature_fn):
    """
    For each subject: compute features per window, then average across windows.
    Returns X:(n_subjects,D), subjects:list
    """
    X, subjects = [], []
    for subj in sorted(subject_to_spike_windows.keys()):
        windows = subject_to_spike_windows[subj]
        feats = [feature_fn(w) for w in windows]
        feats = np.stack(feats, axis=0)
        X.append(np.mean(feats, axis=0))
        subjects.append(subj)
    return np.asarray(X), subjects


# ============================================================
# Metrics (class 1 = positive; class 0 = control)
# ============================================================
def metrics_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()
    tot = tn + fp + fn + tp
    acc = (tp + tn) / tot if tot else np.nan
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    bacc = (sens + spec) / 2 if np.isfinite(sens) and np.isfinite(spec) else np.nan
    return acc, bacc, spec, sens


# ============================================================
# Labels (subject-level)
# ============================================================
subjects = sorted(subject_to_spike_windows.keys())
y = np.array([subject_labels_npz[s] for s in subjects])


# ============================================================
# Feature sets (names match paper)
# ============================================================
feature_methods = [
    ("AFR", afr_window),
    ("TB", lambda sp: tb_window(sp, bins=TB_BINS)),
    ("BI", lambda sp: bi_window(sp, smooth_bins=BI_SMOOTH_BINS)),
    ("AFR+BI", lambda sp: concat_features(afr_window(sp), bi_window(sp, smooth_bins=BI_SMOOTH_BINS))),

    # Topology-aware sets (use the 3x3 structure -> 9 neurons per layer)
    ("L-AFR", lafr_window),
    ("L-TB", lambda sp: ltb_window(sp, bins=TB_BINS)),
    ("L-BI", lambda sp: lbi_window(sp, smooth_bins=BI_SMOOTH_BINS)),
    ("L-AFR+L-BI", lambda sp: concat_features(lafr_window(sp), lbi_window(sp, smooth_bins=BI_SMOOTH_BINS))),

]


# ============================================================
# CV + model pipeline
# ============================================================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=PCA_VAR_THRESHOLD, svd_solver="full")),
    ("svm", SVC(
        kernel=SVM_KERNEL,
        probability=True,
        class_weight=SVM_CLASS_WEIGHT,
        gamma=SVM_GAMMA,
        C=SVM_C
    ))
])

for feat_name, feat_fn in feature_methods:
    X_raw, _ = extract_subject_features(subject_to_spike_windows, feature_fn=feat_fn)

    conf_mats = []
    fold_metrics = []
    pca_dims = []
    for tr, ts in skf.split(X_raw, y):
        model.fit(X_raw[tr], y[tr])
        y_pred = model.predict(X_raw[ts])

        cm = confusion_matrix(y[ts], y_pred, labels=[0, 1])
        conf_mats.append(cm)
        fold_metrics.append(metrics_from_cm(cm))
        pca_dims.append(model.named_steps["pca"].n_components_)

    cm_tot = np.sum(conf_mats, axis=0)
    acc, bacc, spec, sens = metrics_from_cm(cm_tot)

    # ===== Metrics across folds (mean ± std) =====
    fold_metrics = np.array(fold_metrics, dtype=float)
    m = np.nanmean(fold_metrics, axis=0)
    s = np.nanstd(fold_metrics, axis=0)

    # ===== Metrics from aggregated confusion matrix =====
    cm_tot = np.sum(conf_mats, axis=0)
    acc_tot, bacc_tot, spec_tot, sens_tot = metrics_from_cm(cm_tot)

    # ===== Print =====
    print("\n" + "=" * 60)
    print(f"FEATURE SET: {feat_name}")
    print("=" * 60)

    print("Across-fold metrics (mean ± std):")
    print(f"  Accuracy          : {m[0]:.4f} ± {s[0]:.4f}")
    print(f"  Balanced Accuracy : {m[1]:.4f} ± {s[1]:.4f}")
    print(f"  Specificity (TNR) : {m[2]:.4f} ± {s[2]:.4f}")
    print(f"  Sensitivity (TPR) : {m[3]:.4f} ± {s[3]:.4f}")

    print("\nMetrics from aggregated confusion matrix (CM_TOT):")
    print(f"  Accuracy          : {acc_tot:.4f}")
    print(f"  Balanced Accuracy : {bacc_tot:.4f}")
    print(f"  Specificity (TNR) : {spec_tot:.4f}")
    print(f"  Sensitivity (TPR) : {sens_tot:.4f}")
    
    pca_dims = np.array(pca_dims)
    print(f"\nNumber of Feature: {X_raw.shape[1]}")
    print("\nPCA dimensionality:")
    print(f"  Components (mean ± std): {pca_dims.mean():.1f} ± {pca_dims.std():.1f}")
    print(f"  Components (min–max): {pca_dims.min()} – {pca_dims.max()}")
    # Plot aggregated confusion matrix (optional)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm_tot,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["True 0", "True 1"]
    )
    plt.title(f"Confusion Matrix – {feat_name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
  #  plt.show()


In [ ]:
X_raw.shape

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix


# =========================
# Helper: metrics from CM
# =========================
def metrics_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()
    tot = tn + fp + fn + tp
    acc = (tp + tn) / tot if tot else np.nan
    sens = tp / (tp + fn) if (tp + fn) else np.nan          # TPR (class 1)
    spec = tn / (tn + fp) if (tn + fp) else np.nan          # TNR (class 0)
    bacc = (sens + spec) / 2 if np.isfinite(sens) and np.isfinite(spec) else np.nan
    return acc, bacc, spec, sens


# =========================
# Subject-level aggregation
# =========================
def extract_subject_features(subject_to_spike_windows, feature_fn):
    X, subjects = [], []
    for subj in sorted(subject_to_spike_windows.keys()):
        windows = subject_to_spike_windows[subj]
        feats = [feature_fn(w) for w in windows]
        feats = np.stack(feats, axis=0)
        X.append(np.mean(feats, axis=0))
        subjects.append(subj)
    return np.asarray(X), subjects


# =========================
# Model + CV
# =========================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=PCA_VAR_THRESHOLD, svd_solver="full")),
    ("svm", SVC(
        kernel=SVM_KERNEL,
        probability=True,
        class_weight=SVM_CLASS_WEIGHT,
        gamma=SVM_GAMMA,
        C=SVM_C
    ))
])

# =========================
# Run evaluation (NO print, NO plot)
# =========================
results_rows = []
fold_rows = []
cm_total_by_feature = {}

for feat_name, feat_fn in feature_methods:

    X_raw, subj_list = extract_subject_features(subject_to_spike_windows, feature_fn=feat_fn)

    fold_metrics = []
    conf_mats = []

    for fold_idx, (tr, ts) in enumerate(skf.split(X_raw, y), start=1):
        model.fit(X_raw[tr], y[tr])
        y_pred = model.predict(X_raw[ts])

        cm = confusion_matrix(y[ts], y_pred, labels=[0, 1])
        conf_mats.append(cm)

        acc, bacc, spec, sens = metrics_from_cm(cm)
        fold_metrics.append([acc, bacc, spec, sens])

        fold_rows.append({
            "feature": feat_name,
            "fold": fold_idx,
            "accuracy": acc,
            "balanced_accuracy": bacc,
            "specificity": spec,
            "sensitivity": sens
        })

    fold_metrics = np.asarray(fold_metrics, dtype=float)
    m = np.nanmean(fold_metrics, axis=0)
    s = np.nanstd(fold_metrics, axis=0)

    cm_tot = np.sum(conf_mats, axis=0)
    cm_total_by_feature[feat_name] = cm_tot

    results_rows.append({
        "feature": feat_name,
        "accuracy_mean": m[0], "accuracy_std": s[0],
        "balanced_accuracy_mean": m[1], "balanced_accuracy_std": s[1],
        "specificity_mean": m[2], "specificity_std": s[2],
        "sensitivity_mean": m[3], "sensitivity_std": s[3],
        "D": X_raw.shape[1]  # feature dimensionality after extraction (before PCA)
    })

df_results = pd.DataFrame(results_rows).sort_values("balanced_accuracy_mean", ascending=False).reset_index(drop=True)
df_folds = pd.DataFrame(fold_rows)

best_feature = df_results.loc[0, "feature"]  # best by Balanced Accuracy mean


In [ ]:
SAVE_FIGS = True
OUTDIR = "./figures"
import os
os.makedirs(OUTDIR, exist_ok=True)

METRICS = ["accuracy_mean", "balanced_accuracy_mean", "specificity_mean", "sensitivity_mean"]
METRICS_LABELS = ["Accuracy", "Balanced Acc", "Specificity", "Sensitivity"]


In [ ]:
def radar_plot(df, feature_list=None, title="Radar plot (mean metrics)", savepath=None):
    if feature_list is None:
        feature_list = df["feature"].tolist()

    vals = df.set_index("feature").loc[feature_list, METRICS].to_numpy()
    labels = METRICS_LABELS

    n = len(labels)
    angles = np.linspace(0, 2*np.pi, n, endpoint=False).tolist()
    angles += angles[:1]

    fig = plt.figure(figsize=(7, 7))
    ax = plt.subplot(111, polar=True)

    for i, feat in enumerate(feature_list):
        v = vals[i].tolist()
        v += v[:1]
        ax.plot(angles, v, linewidth=1, label=feat)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)
    ax.set_yticks([0.6, 0.7, 0.8, 0.9, 1.0])
    ax.set_ylim(0.5, 1.0)
    ax.set_title(title)

    ax.legend(loc="upper left", bbox_to_anchor=(1.05, 1.0), borderaxespad=0.)
    plt.tight_layout()

    if savepath:
        plt.savefig(savepath, dpi=300, bbox_inches="tight")
    # non plt.show()

# Tutte le feature (può essere molto denso)
radar_plot(
    df_results,
    feature_list=df_results["feature"].tolist(),
    title="Radar plot – all feature sets (mean)",
    savepath=os.path.join(OUTDIR, "radar_all.png") if SAVE_FIGS else None
)

# Versione leggibile: top-3 + 2 baseline
top3 = df_results["feature"].tolist()[:3]
baselines = ["AFR", "TB (20 bins)"]
radar_feats = [f for f in (top3 + baselines) if f in df_results["feature"].tolist()]
radar_plot(
    df_results,
    feature_list=radar_feats,
    title="Radar plot – selected feature sets (mean)",
    savepath=os.path.join(OUTDIR, "radar_selected.png") if SAVE_FIGS else None
)


In [ ]:
def bar_plot_bacc(df, title="Balanced Accuracy (mean ± std)", savepath=None):
    x = np.arange(len(df))
    yv = df["balanced_accuracy_mean"].to_numpy()
    ye = df["balanced_accuracy_std"].to_numpy()

    fig = plt.figure(figsize=(10, 4))
    plt.bar(x, yv, yerr=ye, capsize=3)
    plt.xticks(x, df["feature"].tolist(), rotation=35, ha="right")
    plt.ylim(0.5, 1.0)
    plt.ylabel("Balanced Accuracy")
    plt.title(title)

    # highlight best (outline)
    best_idx = 0
    plt.gca().patches[best_idx].set_linewidth(2)
    plt.gca().patches[best_idx].set_edgecolor("black")

    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=300, bbox_inches="tight")

bar_plot_bacc(
    df_results,
    savepath=os.path.join(OUTDIR, "bar_balanced_accuracy.png") if SAVE_FIGS else None
)


In [ ]:
def scatter_sens_spec(df, title="Sensitivity vs Specificity (mean ± std)", savepath=None):
    x = df["specificity_mean"].to_numpy()
    yv = df["sensitivity_mean"].to_numpy()
    xerr = df["specificity_std"].to_numpy()
    yerr = df["sensitivity_std"].to_numpy()

    fig = plt.figure(figsize=(6, 6))
    plt.errorbar(x, yv, xerr=xerr, yerr=yerr, fmt="o", capsize=3)

    for i, row in df.iterrows():
        plt.text(row["specificity_mean"] + 0.002, row["sensitivity_mean"] + 0.002, row["feature"], fontsize=8)

    plt.xlim(0.5, 1.0)
    plt.ylim(0.5, 1.0)
    plt.xlabel("Specificity (TNR)")
    plt.ylabel("Sensitivity (TPR)")
    plt.title(title)
    plt.grid(True, linewidth=0.3)
    plt.tight_layout()

    if savepath:
        plt.savefig(savepath, dpi=300, bbox_inches="tight")

scatter_sens_spec(
    df_results,
    savepath=os.path.join(OUTDIR, "scatter_sens_vs_spec.png") if SAVE_FIGS else None
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

def key_tradeoff_plot(
    df,
    best_feature,
    title="Trade-off between sensitivity and specificity",
    savepath=None,
    xlim=(0.65, 1.0),
    ylim=(0.65, 1.0),
    tol=1e-6,                  # overlap tolerance (in data space)
    max_label_lines=3,         # if >3 methods overlap, show "+k more"
    label_best=True,
    show_group_labels=True,
    show_legend=True,          # legend shows ALL methods with their colors
    legend_ncol=1,
    legend_loc="lower left"
):
    """
    Paper-ready trade-off plot:
      - One unique color per method (color-blind friendly palette)
      - If multiple methods share the same (x,y) (within tol), plot ONE point
        and annotate with multi-line label (one above another).
      - Highlight best_feature with a black ring.
      - No jitter (truthful): overlap is communicated via joint labels.
    Expected df columns: 'feature', 'specificity_mean', 'sensitivity_mean'
    Optional: 'balanced_accuracy_mean' (only used to choose label order inside overlaps)
    """

    # ---------- extract + clean ----------
    x = df["specificity_mean"].to_numpy(dtype=float)
    y = df["sensitivity_mean"].to_numpy(dtype=float)
    labels = df["feature"].astype(str).to_numpy()

    has_bacc = "balanced_accuracy_mean" in df.columns
    bacc = df["balanced_accuracy_mean"].to_numpy(dtype=float) if has_bacc else None

    m = np.isfinite(x) & np.isfinite(y)
    if has_bacc:
        m = m & np.isfinite(bacc)

    x, y, labels = x[m], y[m], labels[m]
    if has_bacc:
        bacc = bacc[m]

    if len(x) == 0:
        raise ValueError("No finite points to plot (check df columns).")

    # ---------- color map per feature (color-blind safe) ----------
    uniq = sorted(set(labels))  # stable order for legend
    n = len(uniq)
    cmap = plt.get_cmap("tab10" if n <= 10 else "tab20")
    color_map = {f: cmap(i % cmap.N) for i, f in enumerate(uniq)}

    # ---------- group overlapping points ----------
    # Use quantization so points within tol are grouped together
    groups = defaultdict(list)  # key -> list of indices
    qx = np.round(x / tol).astype(np.int64)
    qy = np.round(y / tol).astype(np.int64)

    for i, key in enumerate(zip(qx, qy)):
        groups[key].append(i)

    # ---------- figure ----------
    fig, ax = plt.subplots(figsize=(6.4, 6.0), dpi=300)

    # diagonal (sens=spec)
    ax.plot([xlim[0], xlim[1]], [xlim[0], xlim[1]],
            linestyle="--", linewidth=1.0, color="0.55", zorder=0)

    # ---------- plot each group as ONE point ----------
    for key, idxs in groups.items():
        # representative coordinate (they should be effectively equal)
        xi = float(np.mean(x[idxs]))
        yi = float(np.mean(y[idxs]))

        # choose a representative color:
        # if multiple labels overlap, pick the first by BAcc (if present), else alphabetical
        labs = [labels[i] for i in idxs]
        if has_bacc:
            idxs_sorted = sorted(idxs, key=lambda i: float(bacc[i]), reverse=True)
            rep_lab = labels[idxs_sorted[0]]
        else:
            rep_lab = sorted(labs)[0]

        ax.scatter(
            [xi], [yi],
            s=140,
            color=color_map[rep_lab],
            alpha=0.95,
            edgecolors="white",
            linewidths=0.9,
            zorder=3
        )

        # group label: one above another
        if show_group_labels:
            # order names inside label
            if has_bacc:
                labs_sorted = [labels[i] for i in idxs_sorted]
            else:
                labs_sorted = sorted(labs)

            # shorten if too many overlap
            display = labs_sorted[:max_label_lines]
            extra = len(labs_sorted) - len(display)
            if extra > 0:
                display.append(f"+{extra} more")

            label_text = "\n".join(display)

            ax.annotate(
                label_text,
                xy=(xi, yi),
                xytext=(6, 6),
                textcoords="offset points",
                fontsize=8,
                bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="0.85", alpha=0.9),
                zorder=10
            )

        # highlight ring if best_feature is in this group
        if best_feature in labs:
            ax.scatter(
                [xi], [yi],
                s=260,
                facecolors="none",
                edgecolors="black",
                linewidths=2.2,
                zorder=6
            )
            if label_best and not show_group_labels:
                ax.annotate(
                    best_feature,
                    xy=(xi, yi),
                    xytext=(8, 8),
                    textcoords="offset points",
                    fontsize=9,
                    weight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.75", alpha=0.95),
                    zorder=11
                )

    # ---------- axes styling ----------
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Specificity (TNR)", fontweight='bold')
    ax.set_ylabel("Sensitivity (TPR)", fontweight='bold')
    ax.set_title(title, fontweight='bold')

    ax.grid(True, linewidth=0.35, alpha=0.5)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

    # ---------- legend: one entry per feature ----------
    if show_legend:
        handles = []
        for f in uniq:
            handles.append(
                ax.scatter([], [], s=90, color=color_map[f],
                           edgecolors="white", linewidths=0.9, label=f)
            )
        ax.legend(
            handles=handles,
            loc=legend_loc,
            frameon=False,
            fontsize=8,
            ncol=legend_ncol,
            title="Feature set",
            title_fontsize=9
        )

    fig.tight_layout()

    if savepath:
        fig.savefig(savepath, dpi=300, bbox_inches="tight")

    return fig, ax

SAVE_FIGS = True
print(os.path.join(OUTDIR, "key_tradeoff.png"))
# Example usage:
fig, ax = key_tradeoff_plot(
     df_results,
     best_feature=best_feature,
     title="Trade-off between sensitivity and specificity",
     savepath=os.path.join(OUTDIR, "key_tradeoff.png") if SAVE_FIGS else None,
     show_group_labels=True,
     show_legend=False
 )
plt.savefig(os.path.join(OUTDIR, "key_trade_off.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
def heatmap_metrics(df, title="Metrics heatmap (means)", savepath=None):
    M = df[METRICS].to_numpy()
    fig = plt.figure(figsize=(7, 4))
    plt.imshow(M, aspect="auto")
    plt.yticks(np.arange(len(df)), df["feature"].tolist())
    plt.xticks(np.arange(len(METRICS_LABELS)), METRICS_LABELS, rotation=25, ha="right")
    plt.colorbar(label="score")
    plt.title(title)
    plt.tight_layout()

    if savepath:
        plt.savefig(savepath, dpi=300, bbox_inches="tight")

heatmap_metrics(
    df_results,
    savepath=os.path.join(OUTDIR, "heatmap_metrics.png") if SAVE_FIGS else None
)


In [ ]:
df_results.to_csv(os.path.join(OUTDIR, "results_summary.csv"), index=False)
df_folds.to_csv(os.path.join(OUTDIR, "results_folds.csv"), index=False)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, auc

# =========================
# Params
# =========================
PCA_VAR_THRESHOLD = 0.85
N_SPLITS = 5
RANDOM_STATE = 42

SVM_C = 1.0
SVM_GAMMA = "scale"
SVM_KERNEL = "rbf"
SVM_CLASS_WEIGHT = "balanced"

BI_SMOOTH_BINS = 20

# =========================
# Data (subject-level)
# =========================
subjects = sorted(subject_to_spike_windows.keys())
y = np.array([subject_labels_npz[s] for s in subjects])

X_raw, _ = extract_subject_features(
    subject_to_spike_windows,
    feature_fn=lambda sp: lbi_window(sp, smooth_bins=BI_SMOOTH_BINS)
)

# =========================
# Model
# =========================
model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=PCA_VAR_THRESHOLD, svd_solver="full")),
    ("svm", SVC(
        kernel=SVM_KERNEL,
        probability=False,          # non serve per decision_function
        class_weight=SVM_CLASS_WEIGHT,
        gamma=SVM_GAMMA,
        C=SVM_C
    ))
])

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# =========================
# Collect OUT-OF-FOLD scores
# =========================
y_true_all = []
y_score_all = []

# optional: fold-wise ROC mean±std
mean_fpr = np.linspace(0, 1, 200)
tprs = []
aucs = []

for tr, ts in skf.split(X_raw, y):
    model.fit(X_raw[tr], y[tr])

    scores = model.decision_function(X_raw[ts])  # SVM score (margin)
    y_true_all.append(y[ts])
    y_score_all.append(scores)

    # fold ROC (optional)
    fpr, tpr, _ = roc_curve(y[ts], scores, pos_label=1)
    aucs.append(auc(fpr, tpr))

    tpr_interp = np.interp(mean_fpr, fpr, tpr)
    tpr_interp[0] = 0.0
    tprs.append(tpr_interp)

# concatenate out-of-fold
y_true_all = np.concatenate(y_true_all)
y_score_all = np.concatenate(y_score_all)

# aggregated ROC
fpr_all, tpr_all, _ = roc_curve(y_true_all, y_score_all, pos_label=1)
auc_all = auc(fpr_all, tpr_all)

# mean ± std (optional)
tprs = np.array(tprs)
mean_tpr = np.mean(tprs, axis=0)
std_tpr = np.std(tprs, axis=0)
mean_tpr[-1] = 1.0

mean_auc = float(np.mean(aucs))
std_auc = float(np.std(aucs))

# =========================
# Plot
# =========================
plt.figure(figsize=(6, 5))
plt.plot(fpr_all, tpr_all, linewidth=2, label=f"ROC (OOF agg) AUC={auc_all:.3f}")

# optional mean ROC + band
plt.plot(mean_fpr, mean_tpr, linestyle="--", linewidth=2,
         label=f"Mean ROC (folds) AUC={mean_auc:.3f}±{std_auc:.3f}")
plt.fill_between(
    mean_fpr,
    np.clip(mean_tpr - std_tpr, 0, 1),
    np.clip(mean_tpr + std_tpr, 0, 1),
    alpha=0.2,
    label="±1 std (folds)"
)

plt.plot([0, 1], [0, 1], linestyle=":", linewidth=1, color="gray", label="Chance")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC – L-BI (layer pooled) [SVM decision_function]")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"AUC aggregated (out-of-fold): {auc_all:.4f}")
print(f"AUC mean±std across folds: {mean_auc:.4f} ± {std_auc:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, auc

# =========================
# Parameters
# =========================
PCA_VAR_THRESHOLD = 0.85
N_SPLITS = 5
RANDOM_STATE = 42

SVM_C = 1.0
SVM_GAMMA = "scale"
SVM_KERNEL = "rbf"
SVM_CLASS_WEIGHT = "balanced"

BI_SMOOTH_BINS = 20

# =========================
# Labels (subject-level)
# =========================
subjects = sorted(subject_to_spike_windows.keys())
y = np.array([subject_labels_npz[s] for s in subjects])

# =========================
# L-BI features (layer pooled)
# =========================
X, _ = extract_subject_features(
    subject_to_spike_windows,
    feature_fn=lambda sp: lbi_window(sp, smooth_bins=BI_SMOOTH_BINS)
)

# =========================
# Model
# =========================
model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=PCA_VAR_THRESHOLD, svd_solver="full")),
    ("svm", SVC(
        kernel=SVM_KERNEL,
        probability=True,
        class_weight=SVM_CLASS_WEIGHT,
        gamma=SVM_GAMMA,
        C=SVM_C
    ))
])

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# =========================
# Collect OUT-OF-FOLD scores
# =========================
y_true_all = []
y_score_all = []

for tr, ts in skf.split(X, y):
    model.fit(X[tr], y[tr])
    prob = model.predict_proba(X[ts])[:, 1]  # positive class
    y_true_all.append(y[ts])
    y_score_all.append(prob)

y_true_all = np.concatenate(y_true_all)
y_score_all = np.concatenate(y_score_all)

# =========================
# ROC + AUC
# =========================
fpr, tpr, _ = roc_curve(y_true_all, y_score_all, pos_label=1)
roc_auc = auc(fpr, tpr)

# =========================
# Plot (paper style)
# =========================
plt.figure(figsize=(5.5, 5))
plt.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", lw=1, color="gray")

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (1 − Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC – L-BI (layer pooled)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"AUC (out-of-fold, aggregated): {roc_auc:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# fpr, tpr già calcolati
fpr, tpr, _ = roc_curve(y_true_all, y_score_all, pos_label=1)
roc_auc = auc(fpr, tpr)

# smoothing solo grafico
fpr_grid = np.linspace(0, 1, 400)
tpr_smooth = np.interp(fpr_grid, fpr, tpr)
tpr_smooth[0] = 0.0
tpr_smooth[-1] = 1.0

plt.figure(figsize=(5.5, 5))
plt.plot(fpr_grid, tpr_smooth, lw=2, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], "--", lw=1, color="gray")
plt.xlabel("False Positive Rate (1 − Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC – L-BI (layer pooled)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, confusion_matrix

def pick_threshold_from_train(y_train, score_train, pos_label, criterion="bacc", max_fpr=None):
    """
    Choose a threshold using ONLY training data.
    score_train: continuous score (e.g., decision_function)
    pos_label: which label is considered "positive" (must match y_train)
    criterion: "youden" | "bacc" | "fpr"
    """
    fpr, tpr, thr = roc_curve(y_train, score_train, pos_label=pos_label)

    if criterion == "youden":
        j = tpr - fpr
        return thr[int(np.nanargmax(j))]

    if criterion == "bacc":
        tnr = 1.0 - fpr
        bacc = 0.5 * (tpr + tnr)
        return thr[int(np.nanargmax(bacc))]

    if criterion == "fpr":
        if max_fpr is None:
            raise ValueError("Set max_fpr when criterion='fpr'")
        valid = np.where(fpr <= max_fpr)[0]
        if len(valid) == 0:
            return thr[int(np.nanargmin(fpr))]
        best = valid[int(np.nanargmax(tpr[valid]))]
        return thr[best]

    raise ValueError("criterion must be 'youden', 'bacc', or 'fpr'")

def metrics_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) else np.nan  # TPR
    spec = tn / (tn + fp) if (tn + fp) else np.nan  # TNR
    bacc = 0.5 * (sens + spec) if np.isfinite(sens) and np.isfinite(spec) else np.nan
    return sens, spec, bacc

# =========================
# Thresholded CV (SVM scores)
# =========================
criterion = "bacc"   # "youden" | "bacc" | "fpr"
max_fpr = 0.05       # used only if criterion="fpr"

thresholds = []
cms = []

for tr, ts in skf.split(X, y):
    model.fit(X[tr], y[tr])

    # SVM scores (margin)
    s_tr = model.decision_function(X[tr])
    s_ts = model.decision_function(X[ts])

    # which class is "positive" for this fitted model
    classes = model.named_steps["svm"].classes_
    pos_label = classes[1]
    neg_label = classes[0]

    # threshold chosen on TRAIN only
    thr = pick_threshold_from_train(y[tr], s_tr, pos_label=pos_label,
                                    criterion=criterion, max_fpr=max_fpr)
    thresholds.append(thr)

    # apply threshold on TEST (map back to labels)
    y_pred = np.where(s_ts >= thr, pos_label, neg_label)

    cm = confusion_matrix(y[ts], y_pred, labels=[neg_label, pos_label])
    cms.append(cm)

cm_tot = np.sum(cms, axis=0)
sens, spec, bacc = metrics_from_cm(cm_tot)

print("====================================")
print(f"Threshold criterion: {criterion}")
if criterion == "fpr":
    print(f"Max FPR constraint: {max_fpr}")
print("====================================")
print("Thresholds per fold:", np.round(thresholds, 4))
print(f"Mean threshold: {np.mean(thresholds):.4f} ± {np.std(thresholds):.4f}")
print("\nAggregated CM (sum over folds):")
print(cm_tot)
print(f"\nSensitivity (TPR): {sens:.4f}")
print(f"Specificity (TNR): {spec:.4f}")
print(f"Balanced Accuracy: {bacc:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# ============================================================
# THRESHOLD ANALYSIS on OUT-OF-FOLD SVM SCORES (subject-level)
# ============================================================
# Assumptions:
# - X: (n_subjects, D) subject-level features (e.g., L-BI pooled)
# - y: (n_subjects,) labels in {0,1}
# - skf: StratifiedKFold instance (same you used)
# - model: your Pipeline with SVC inside (probability NOT needed)
#
# This script:
# 1) collects out-of-fold decision_function scores for each subject
# 2) sweeps thresholds and computes Sens, Spec, FPR, BAcc
# 3) finds: tau=0 (SVC predict), tau maximizing BAcc, tau with FPR<=alpha
# 4) plots metrics vs threshold + prints confusion matrices at key taus
# ============================================================

def cm_and_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sens = tp / (tp + fn) if (tp + fn) else np.nan  # TPR
    spec = tn / (tn + fp) if (tn + fp) else np.nan  # TNR
    fpr  = fp / (fp + tn) if (fp + tn) else np.nan  # 1 - spec
    bacc = 0.5 * (sens + spec) if np.isfinite(sens) and np.isfinite(spec) else np.nan

    return cm, sens, spec, fpr, bacc

# ----------------------------
# 1) Collect OOF scores
# ----------------------------
n = len(y)
oof_score = np.empty(n, dtype=float)

for tr, ts in skf.split(X, y):
    model.fit(X[tr], y[tr])
    oof_score[ts] = model.decision_function(X[ts])

# ----------------------------
# 2) Threshold sweep
# ----------------------------
# Use a dense grid + also include exact observed scores (useful with small N)
grid = np.linspace(oof_score.min(), oof_score.max(), 400)
unique_scores = np.unique(oof_score)
thresholds = np.unique(np.concatenate([grid, unique_scores]))

sens_list, spec_list, fpr_list, bacc_list = [], [], [], []

for thr in thresholds:
    y_pred = (oof_score >= thr).astype(int)
    _, sens, spec, fpr, bacc = cm_and_metrics(y, y_pred)
    sens_list.append(sens)
    spec_list.append(spec)
    fpr_list.append(fpr)
    bacc_list.append(bacc)

sens_arr = np.array(sens_list, dtype=float)
spec_arr = np.array(spec_list, dtype=float)
fpr_arr  = np.array(fpr_list, dtype=float)
bacc_arr = np.array(bacc_list, dtype=float)

# ----------------------------
# 3) Key thresholds
# ----------------------------
# (a) SVC predict threshold in score-space is tau = 0
tau_predict = 0.0

# (b) threshold that maximizes Balanced Accuracy (descriptive)
idx_bacc = int(np.nanargmax(bacc_arr))
tau_bacc = thresholds[idx_bacc]

# (c) threshold under a max-FPR constraint (choose best sensitivity)
alpha = 0.05  # change if you want (e.g., 0.10)
valid = np.where(fpr_arr <= alpha)[0]
if len(valid) > 0:
    best = valid[int(np.nanargmax(sens_arr[valid]))]
    tau_fpr = thresholds[best]
else:
    tau_fpr = None

# ----------------------------
# 4) Print summary + CMs
# ----------------------------
print("====================================")
print("OUT-OF-FOLD Threshold Analysis (SVM decision_function)")
print("====================================")
print(f"Score range: [{oof_score.min():.4f}, {oof_score.max():.4f}]")
print(f"tau_predict (SVC default): {tau_predict:.4f}")
print(f"tau_maxBAcc: {tau_bacc:.4f}  (BAcc={bacc_arr[idx_bacc]:.4f}, Sens={sens_arr[idx_bacc]:.4f}, Spec={spec_arr[idx_bacc]:.4f})")
if tau_fpr is not None:
    j = np.where(thresholds == tau_fpr)[0][0]
    print(f"tau_FPR<= {alpha:.2f}: {tau_fpr:.4f}  (FPR={fpr_arr[j]:.4f}, Sens={sens_arr[j]:.4f}, Spec={spec_arr[j]:.4f}, BAcc={bacc_arr[j]:.4f})")
else:
    print(f"No threshold achieves FPR <= {alpha:.2f}")

def report_at_tau(tau, name):
    y_pred = (oof_score >= tau).astype(int)
    cm, sens, spec, fpr, bacc = cm_and_metrics(y, y_pred)
    print("\n" + "-"*50)
    print(f"{name}  (tau={tau:.4f})")
    print("CM [[TN FP],[FN TP]]:")
    print(cm)
    print(f"Sens={sens:.4f}  Spec={spec:.4f}  FPR={fpr:.4f}  BAcc={bacc:.4f}")

report_at_tau(tau_predict, "SVC predict (default margin)")

report_at_tau(tau_bacc, "Max Balanced Accuracy (descriptive)")

if tau_fpr is not None:
    report_at_tau(tau_fpr, f"Best Sens with FPR<= {alpha:.2f}")

# ----------------------------
#


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# ============================================================
# ROC from OUT-OF-FOLD scores (decision_function)
# ============================================================
# Assumptions:
# - You already computed:
#   oof_score: (n_subjects,) out-of-fold SVM decision_function scores
#   y (or oof_true): (n_subjects,) true labels in {0,1}
# - You already have (optional) threshold analysis arrays:
#   thresholds, sens_arr, spec_arr, fpr_arr, bacc_arr
# - And key thresholds:
#   tau_predict (=0), tau_bacc, tau_fpr (optional)
# ============================================================

# If you used oof_true name in your script, set:
oof_true = y

# --- ROC + AUC ---
fpr, tpr, roc_thr = roc_curve(oof_true, oof_score, pos_label=1)
roc_auc = auc(fpr, tpr)

# --- Plot ROC ---
plt.figure(figsize=(5.8, 5.2))
plt.plot(fpr, tpr, lw=2, label=f"ROC (OOF)  AUC={roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", lw=1, color="gray", label="Chance")

# --- OPTIONAL: mark operating points from your threshold analysis ---
def point_at_tau(tau):
    # find closest threshold in your sweep (works even if tau not exactly in thresholds)
    idx = int(np.argmin(np.abs(thresholds - tau)))
    return float(fpr_arr[idx]), float(sens_arr[idx]), float(thresholds[idx])

# Mark τ = 0 (SVC predict)
if "thresholds" in globals():
    x0, y0, tau0_used = point_at_tau(0.0)
    plt.scatter([x0], [y0], s=45, marker="o", color="black",
                label=f"τ=0 (SVC predict) [≈{tau0_used:.3f}]")

    # Mark τ that maximizes BAcc (if you computed it)
    if "tau_bacc" in globals():
        xb, yb, taub_used = point_at_tau(tau_bacc)
        plt.scatter([xb], [yb], s=45, marker="s", color="red",
                    label=f"τ max BAcc [≈{taub_used:.3f}]")

    # Mark τ under FPR constraint (if you computed it)
    if "tau_fpr" in globals() and tau_fpr is not None:
        xf, yf, tauf_used = point_at_tau(tau_fpr)
        plt.scatter([xf], [yf], s=55, marker="^", color="green",
                    label=f"τ (FPR≤α) [≈{tauf_used:.3f}]")

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (1 − Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC – Out-of-fold SVM scores (decision_function)")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

print(f"AUC (OOF): {roc_auc:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix

# X, y, skf, model devono esistere già

# ----------------------------
# 1) Collect OOF scores (SAFE)
# ----------------------------
n = len(y)
oof_score = np.full(n, np.nan, dtype=float)   # NaN guard
oof_pred  = np.full(n, -1, dtype=int)         # guard

for fold, (tr, ts) in enumerate(skf.split(X, y), start=1):
    model.fit(X[tr], y[tr])

    scores_ts = model.decision_function(X[ts])
    preds_ts  = model.predict(X[ts])

    # critical: write ONLY into the test indices
    oof_score[ts] = scores_ts
    oof_pred[ts]  = preds_ts

    # quick sanity check per fold (optional)
    # print(f"Fold {fold}: wrote {len(ts)} test scores")

# sanity checks
if np.any(np.isnan(oof_score)):
    missing = np.where(np.isnan(oof_score))[0]
    raise RuntimeError(f"OOF scores not filled for indices: {missing.tolist()}")

if np.any(oof_pred == -1):
    missing = np.where(oof_pred == -1)[0]
    raise RuntimeError(f"OOF preds not filled for indices: {missing.tolist()}")

# ----------------------------
# 2) ROC + AUC from OOF scores
# ----------------------------
fpr, tpr, _ = roc_curve(y, oof_score, pos_label=1)
roc_auc = auc(fpr, tpr)

# ----------------------------
# 3) Plot ROC
# ----------------------------
plt.figure(figsize=(5.8, 5.2))
plt.plot(fpr, tpr, lw=2, label=f"ROC (OOF)  AUC={roc_auc:.3f}")
plt.plot([0, 1], [0, 1], "--", lw=1, color="gray", label="Chance")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (1 − Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC – Out-of-fold SVM scores (decision_function)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"AUC (OOF): {roc_auc:.4f}")

# ----------------------------
# 4) (Optional) Show OOF CM using SVC default decision rule
# ----------------------------
cm = confusion_matrix(y, oof_pred, labels=[0, 1])
print("OOF confusion matrix [[TN FP],[FN TP]]:")
print(cm)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix

# ------------------------------------------------------------
# Assumptions:
# - oof_score : (n,) OOF decision_function scores
# - y         : (n,) true labels {0,1}
# ------------------------------------------------------------

# ============== ROC (OOF) ==============
fpr, tpr, roc_thr = roc_curve(y, oof_score, pos_label=1)
roc_auc = auc(fpr, tpr)

# ============== Threshold sweep (from OOF scores) ==============
thresholds = np.unique(oof_score)  # discrete is fine here

sens_list, spec_list, fpr_list, bacc_list = [], [], [], []

for thr in thresholds:
    y_pred = (oof_score >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0,1]).ravel()

    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    fpr_v = 1.0 - spec
    bacc = 0.5 * (sens + spec)

    sens_list.append(sens)
    spec_list.append(spec)
    fpr_list.append(fpr_v)
    bacc_list.append(bacc)

sens_arr = np.array(sens_list)
spec_arr = np.array(spec_list)
fpr_arr  = np.array(fpr_list)
bacc_arr = np.array(bacc_list)

# ============== Operating points ==============
# (1) SVC default: tau = 0
def idx_nearest_tau(tau):
    return int(np.argmin(np.abs(thresholds - tau)))

idx_tau0 = idx_nearest_tau(0.0)

# (2) Max Balanced Accuracy (descriptive)
idx_bacc = int(np.argmax(bacc_arr))

# (3) Best sensitivity under FPR constraint (optional)
alpha = 0.05
valid = np.where(fpr_arr <= alpha)[0]
idx_fpr = valid[int(np.argmax(sens_arr[valid]))] if len(valid) > 0 else None

# ============== Plot ROC + points ==============
plt.figure(figsize=(5.8, 5.2))
plt.plot(fpr, tpr, lw=2, label=f"ROC (OOF)  AUC={roc_auc:.3f}")
plt.plot([0, 1], [0, 1], "--", lw=1, color="gray", label="Chance")

# τ = 0 (SVC predict)
plt.scatter(
    fpr_arr[idx_tau0], sens_arr[idx_tau0],
    s=60, color="black", marker="o",
    label=f"τ=0 (SVC predict)\nBAcc={bacc_arr[idx_tau0]:.2f}"
)

# τ max BAcc
plt.scatter(
    fpr_arr[idx_bacc], sens_arr[idx_bacc],
    s=60, color="red", marker="s",
    label=f"τ max BAcc\n(BAcc={bacc_arr[idx_bacc]:.2f})"
)

# τ with FPR <= alpha (optional)
if idx_fpr is not None:
    plt.scatter(
        fpr_arr[idx_fpr], sens_arr[idx_fpr],
        s=70, color="green", marker="^",
        label=f"τ FPR≤{alpha}\n(BAcc={bacc_arr[idx_fpr]:.2f})"
    )

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (1 − Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC with operating points (out-of-fold scores)")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

print(f"AUC (OOF): {roc_auc:.4f}")

# ============== (Optional) Print CMs at those taus ==============
def print_cm_at(idx, name):
    thr = thresholds[idx]
    y_pred = (oof_score >= thr).astype(int)
    cm = confusion_matrix(y, y_pred, labels=[0,1])
    print("\n" + "-"*40)
    print(f"{name}  (tau={thr:.4f})")
    print("CM [[TN FP],[FN TP]]:")
    print(cm)

print_cm_at(idx_tau0, "SVC predict (tau=0)")
print_cm_at(idx_bacc, "Max Balanced Accuracy")
if idx_fpr is not None:
    print_cm_at(idx_fpr, f"FPR ≤ {alpha}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix

# ============================
# Helpers
# ============================
def metrics_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    bacc = 0.5 * (sens + spec) if np.isfinite(sens) and np.isfinite(spec) else np.nan
    fpr  = 1.0 - spec if np.isfinite(spec) else np.nan
    return sens, spec, bacc, fpr

# ============================
# ROC from OOF scores
# ============================
fpr, tpr, _ = roc_curve(y, oof_score, pos_label=1)
roc_auc = auc(fpr, tpr)

# ============================
# Operating point 1: SVM decision (predict)
# ============================
y_pred_svm = (oof_score >= 0).astype(int)
cm_svm = confusion_matrix(y, y_pred_svm, labels=[0, 1])
sens_svm, spec_svm, bacc_svm, fpr_svm = metrics_from_cm(cm_svm)

# ============================
# Operating point 2: Best Balanced Accuracy (OOF, descriptive)
# ============================
thresholds = np.unique(oof_score)
best_idx = None
best_bacc = -1.0

for thr in thresholds:
    y_pred = (oof_score >= thr).astype(int)
    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    sens, spec, bacc, fpr_v = metrics_from_cm(cm)

    if np.isfinite(bacc) and bacc > best_bacc:
        best_bacc = bacc
        best_idx = thr
        best_vals = (sens, spec, bacc, fpr_v)
        best_cm = cm

sens_b, spec_b, bacc_b, fpr_b = best_vals

# ============================
# Plot — IEEE / EMBC style (color-blind safe)
# ============================
# Okabe–Ito palette
C_BLUE   = "#0072B2"
C_ORANGE = "#E69F00"
C_GREEN  = "#009E73"
C_GRAY   = "#7A7A7A"

plt.figure(figsize=(5.6, 5.2), dpi=300)

# ROC
plt.plot(fpr, tpr, lw=2.5, color=C_BLUE, label=f"ROC (AUC = {roc_auc:.2f})")

# Chance
plt.plot([0, 1], [0, 1], "--", lw=1.2, color=C_GRAY) #, label="Chance")

# SVM decision point
plt.scatter(
    fpr_svm, sens_svm,
    s=70, marker="o", color=C_ORANGE,
    edgecolor="black", linewidth=0.6,
    label=f"SVM decision (BAcc = {bacc_svm:.2f})"
)

# Best BAcc point
plt.scatter(
    fpr_b, sens_b,
    s=85, marker="s", color=C_GREEN,
    edgecolor="black", linewidth=0.6,
    label=f"Best BAcc (BAcc = {bacc_b:.2f})"
)

plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.xlabel("False Positive Rate (1 − Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC curve (L-BI) ")
plt.legend(loc="lower right", frameon=False, fontsize=9)
plt.tight_layout()
#plt.show()
plt.savefig('ROC.png')

# ============================
# Print summary (for Results)
# ============================
print(f"AUC (out-of-fold): {roc_auc:.4f}")

print("\nSVM decision:")
print(f"  Sensitivity: {sens_svm:.4f}")
print(f"  Specificity: {spec_svm:.4f}")
print(f"  Balanced Accuracy: {bacc_svm:.4f}")
print("  Confusion matrix [[TN FP],[FN TP]]:")
print(cm_svm)

print("\nBest Balanced Accuracy (descriptive, OOF):")
print(f"  Sensitivity: {sens_b:.4f}")
print(f"  Specificity: {spec_b:.4f}")
print(f"  Balanced Accuracy: {bacc_b:.4f}")
print("  Confusion matrix [[TN FP],[FN TP]]:")
print(best_cm)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix

# ============================
# Helpers
# ============================
def metrics_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    bacc = 0.5 * (sens + spec) if np.isfinite(sens) and np.isfinite(spec) else np.nan
    fpr  = 1.0 - spec if np.isfinite(spec) else np.nan
    return sens, spec, bacc, fpr

def best_bacc_threshold(y_true, scores):
    thr = np.unique(scores)
    best_thr, best_bacc, best_cm, best_vals = None, -1.0, None, None
    for t in thr:
        y_pred = (scores >= t).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        sens, spec, bacc, fpr_v = metrics_from_cm(cm)
        if np.isfinite(bacc) and bacc > best_bacc:
            best_bacc = bacc
            best_thr = t
            best_cm = cm
            best_vals = (sens, spec, bacc, fpr_v)
    return best_thr, best_cm, best_vals  # best_vals=(sens,spec,bacc,fpr)

# ============================
# 1) Aggregated ROC (OOF)
# ============================
fpr_all, tpr_all, _ = roc_curve(y, oof_score, pos_label=1)
auc_all = auc(fpr_all, tpr_all)

# operating point: SVM decision (tau=0)
y_pred_svm = (oof_score >= 0).astype(int)
cm_svm = confusion_matrix(y, y_pred_svm, labels=[0, 1])
sens_svm, spec_svm, bacc_svm, fpr_svm = metrics_from_cm(cm_svm)

# operating point: best BAcc on OOF scores (descriptive)
thr_b, cm_b, (sens_b, spec_b, bacc_b, fpr_b) = best_bacc_threshold(y, oof_score)

# ============================
# 2) Fold-wise ROC + fold-wise BAcc (mean±std)
#    Requires: skf, X, y, model already defined
# ============================
mean_fpr = np.linspace(0, 1, 300)
tprs = []
aucs = []
bacc_fold_svm = []
bacc_fold_best = []

for tr, ts in skf.split(X, y):
    model.fit(X[tr], y[tr])

    # scores on test fold
    scores_ts = model.decision_function(X[ts])

    # fold ROC/AUC
    fpr, tpr, _ = roc_curve(y[ts], scores_ts, pos_label=1)
    aucs.append(auc(fpr, tpr))

    # interpolate for mean ROC
    tpr_i = np.interp(mean_fpr, fpr, tpr)
    tpr_i[0] = 0.0
    tprs.append(tpr_i)

    # fold operating point: SVM decision (tau=0)
    y_pred0 = (scores_ts >= 0).astype(int)
    cm0 = confusion_matrix(y[ts], y_pred0, labels=[0, 1])
    _, _, bacc0, _ = metrics_from_cm(cm0)
    bacc_fold_svm.append(bacc0)

    # fold operating point: best BAcc threshold (chosen on *this fold's test* -> descriptive only)
    # NOTE: this is NOT for model selection, just to show what "best achievable" on that fold looks like.
    # If you want a proper threshold selection, do nested CV.
    _, cm_best, (_, _, bacc_best, _) = best_bacc_threshold(y[ts], scores_ts)
    bacc_fold_best.append(bacc_best)

tprs = np.array(tprs, dtype=float)
mean_tpr = np.nanmean(tprs, axis=0)
std_tpr = np.nanstd(tprs, axis=0)
mean_tpr[-1] = 1.0

mean_auc = float(np.nanmean(aucs))
std_auc  = float(np.nanstd(aucs))

bacc_svm_mean = float(np.nanmean(bacc_fold_svm))
bacc_svm_std  = float(np.nanstd(bacc_fold_svm))
bacc_best_mean = float(np.nanmean(bacc_fold_best))
bacc_best_std  = float(np.nanstd(bacc_fold_best))

# ============================
# Plot — IEEE / EMBC style (color-blind safe)
# ============================
C_AGG_ROC   = "#0072B2"  # blue  – ROC aggregata (main result)
C_MEAN_ROC  = "#D55E00"  # vermillion – ROC media sui fold
C_STD_BAND  = "#D55E00"  # same, trasparente
C_SVM_PT    = "#E69F00"  # orange – SVM decision
C_BEST_PT   = "#009E73"  # green – Best BAcc
C_CHANCE   = "#7A7A7A"  # gray – chance
plt.figure(figsize=(5.9, 5.4), dpi=300)

# --- Aggregated ROC (OOF) ---
plt.plot(
    fpr_all, tpr_all,
    lw=2.8, color=C_AGG_ROC,
    label=f"ROC (Aggregated)  AUC={auc_all:.2f}"
)

# --- Mean ROC across folds ---
plt.plot(
    mean_fpr, mean_tpr,
    lw=2.2, linestyle="--", color=C_MEAN_ROC,
    label=f"ROC (Fold mean)  AUC={mean_auc:.2f}±{std_auc:.2f}"
)

# --- Variability band ---
#plt.fill_between(
#    mean_fpr,
#    np.clip(mean_tpr - std_tpr, 0, 1),
#    np.clip(mean_tpr + std_tpr, 0, 1),
#    color=C_STD_BAND, alpha=0.22, linewidth=0
#)

# --- Chance level ---
plt.plot(
    [0, 1], [0, 1],
    linestyle=":", lw=1.4, color=C_CHANCE,
 #   label="Chance"
)

# --- Operating point: SVM decision ---
plt.scatter(
    fpr_svm, sens_svm,
    s=85, marker="o",
    color=C_SVM_PT,
    edgecolor="black", linewidth=0.7,
    zorder=5,
    label=f"SVM decision  (BAcc={bacc_svm:.2f})"
)

# --- Operating point: Best BAcc ---
plt.scatter(
    fpr_b, sens_b,
    s=95, marker="s",
    color=C_BEST_PT,
    edgecolor="black", linewidth=0.7,
    zorder=6,
    label=f"Best BAcc (BAcc={bacc_b:.2f})"
)

# --- Axes & layout ---
plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.xlabel("False Positive Rate (1 − Specificity)", fontweight='bold', fontsize=14)
plt.ylabel("True Positive Rate (Sensitivity)", fontweight='bold', fontsize=14)
plt.title("ROC curve - L-BI", fontweight='bold', fontsize=18)

plt.legend(
    loc="lower right",
    frameon=False,
    fontsize=10
)

plt.tight_layout()
plt.savefig("ROC_wMean.png", dpi=300)
# plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix

# =========================
# CONFIG (keep yours)
# =========================
PCA_VAR_THRESHOLD = 0.85   # or your chosen value (0.5 etc.)
N_SPLITS = 5
RANDOM_STATE = 42

SVM_C = 1.0
SVM_GAMMA = "scale"
SVM_KERNEL = "rbf"
SVM_CLASS_WEIGHT = "balanced"

BI_SMOOTH_BINS = 20

# =========================
# ASSUMES you already have:
# - subject_to_spike_windows
# - subject_labels_npz
# - extract_subject_features(...)
# - lbi_window(...)
# =========================

# ---- labels in subject order consistent with X ----
subjects = sorted(subject_to_spike_windows.keys())
y = np.array([subject_labels_npz[s] for s in subjects])

# ---- ONLY L-BI features ----
X_raw, _ = extract_subject_features(
    subject_to_spike_windows,
    feature_fn=lambda sp: lbi_window(sp, smooth_bins=BI_SMOOTH_BINS)
)

# ---- pipeline ----
model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=PCA_VAR_THRESHOLD, svd_solver="full")),
    ("svm", SVC(
        kernel=SVM_KERNEL,
        probability=False,
        class_weight=SVM_CLASS_WEIGHT,
        gamma=SVM_GAMMA,
        C=SVM_C
    ))
])

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# =========================
# Aggregate confusion matrix across folds (OOF predictions)
# =========================
cms = []
for tr, ts in skf.split(X_raw, y):
    model.fit(X_raw[tr], y[tr])
    y_pred = model.predict(X_raw[ts])
    cms.append(confusion_matrix(y[ts], y_pred, labels=[0, 1]))



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import PowerNorm, LinearSegmentedColormap

norm = PowerNorm(gamma=0.7, vmin=0, vmax=1)

cm_tot = np.sum(cms, axis=0)                     
cm_norm = cm_tot / cm_tot.sum(axis=1, keepdims=True)

plt.figure(figsize=(5.0, 4.2), dpi=300)

# ---- build a true Matplotlib colormap (truncated Blues) ----
base = sns.color_palette("Blues", as_cmap=True)  # this is a real colormap
colors = base(np.linspace(0.03, 0.98, 256))      # tweak endpoints: whiter lows, darker highs
cmap = LinearSegmentedColormap.from_list("Blues_trunc", colors)

ax = sns.heatmap(
    cm_norm,
    annot=cm_tot,
    fmt="d",
    norm=norm,                 # keeps small values visible
    cmap=cmap,                 # continuous colormap
    square=True,
    linewidths=0.79,
    linecolor="black",
    cbar=True,
   # cbar_kws=dict(shrink=0.85),
)

ax.set_title("Confusion Matrix (L-BI)", pad=10, fontweight="bold", fontsize=18)
ax.set_xlabel("Predicted label", fontweight="bold")
ax.set_ylabel("True label", fontweight="bold")

ax.set_xticklabels(["TD", "CP"], rotation=0, fontweight="bold", fontsize=14)
ax.set_yticklabels(["TD", "CP"], rotation=90, fontweight="bold", fontsize=14)

for t in ax.texts:
    t.set_fontsize(11)
    t.set_fontweight("bold")
  #  t.set_color("black")

# colorbar: keep it continuous but show only endpoints
cbar = ax.collections[0].colorbar
cbar.set_ticks([0, 1])
cbar.set_ticklabels(["0%", "100%"], fontweight='bold')
cbar.ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig("CM_LBI_PAPER.png", dpi=300)
plt.show()

print("Aggregated confusion matrix (counts):")
print(cm_tot)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# =========================
# ASSUMES you already have:
# - subject_to_spike_windows
# - subject_labels_npz
# - extract_subject_features(...)
# - lbi_window(...)
# =========================

BI_SMOOTH_BINS = 20

# ---- labels in subject order consistent with X ----
subjects = sorted(subject_to_spike_windows.keys())
y = np.array([subject_labels_npz[s] for s in subjects])

# ---- ONLY L-BI features ----
X_raw, _ = extract_subject_features(
    subject_to_spike_windows,
    feature_fn=lambda sp: lbi_window(sp, smooth_bins=BI_SMOOTH_BINS)
)

# =========================
# PCA (2D) for visualization
# =========================
scaler = StandardScaler()
Xz = scaler.fit_transform(X_raw)

pca = PCA(n_components=2, svd_solver="full", random_state=42)
Xp = pca.fit_transform(Xz)
evr = pca.explained_variance_ratio_

# =========================
# Plot (slide-friendly)
# =========================
# Label names (edit to match your narrative)
label_name = {0: "TD", 1: "CP"}   # or {0:"Non-CP", 1:"CP"}
markers = {0: "o", 1: "s"}        # circle vs square

plt.figure(figsize=(9.5, 6.0))
ax = plt.gca()

# Main scatter by class
for lab in sorted(np.unique(y)):
    idx = (y == lab)
    ax.scatter(
        Xp[idx, 0], Xp[idx, 1],
        s=80, marker=markers.get(lab, "o"),
        alpha=0.85, edgecolors="white", linewidths=0.8,
        label=label_name.get(lab, str(lab))
    )

# Optional: show class centroids (good for storytelling)
for lab in sorted(np.unique(y)):
    idx = (y == lab)
    c = Xp[idx].mean(axis=0)
    ax.scatter(
        [c[0]], [c[1]],
        s=220, marker="X", linewidths=2.0,
        label=f"{label_name.get(lab, lab)} centroid"
    )

# Optional: 95% covariance ellipse per class (nice “scientific comms” cue)
# (No external libs; pure matplotlib)
def add_cov_ellipse(points, n_std=2.4477, **kwargs):
    # n_std=2.4477 ~ sqrt(chi2.ppf(0.95, df=2)) for 95% ellipse in 2D
    # Without scipy, we hardcode this common value.
    if points.shape[0] < 3:
        return
    cov = np.cov(points.T)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    width, height = 2 * n_std * np.sqrt(np.maximum(vals, 1e-12))

    from matplotlib.patches import Ellipse
    mean = points.mean(axis=0)
    ell = Ellipse(xy=mean, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ell)

for lab in sorted(np.unique(y)):
    idx = (y == lab)
    add_cov_ellipse(
        Xp[idx],
        facecolor="none",
        linewidth=2.0,
        alpha=0.9
    )

# Titles/labels optimized for a slide
ax.set_title("PCA of speech-dynamics features — subject-level", pad=14)
ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({evr[1]*100:.1f}% variance)")

# Clean grid + spines (readable on projector)
ax.grid(True, alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Legend outside, avoids covering points
ax.legend(
    frameon=False,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0.0
)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3D)

# =========================
# ASSUMES you already have:
# - subject_to_spike_windows
# - subject_labels_npz
# - extract_subject_features(...)
# - lbi_window(...)
# =========================

BI_SMOOTH_BINS = 20

# ---- labels in subject order consistent with X ----
subjects = sorted(subject_to_spike_windows.keys())
y = np.array([subject_labels_npz[s] for s in subjects])

# ---- ONLY L-BI features ----
X_raw, _ = extract_subject_features(
    subject_to_spike_windows,
    feature_fn=lambda sp: lbi_window(sp, smooth_bins=BI_SMOOTH_BINS)
)

# =========================
# PCA (3D) for visualization
# =========================
Xz = StandardScaler().fit_transform(X_raw)

pca = PCA(n_components=3, svd_solver="full", random_state=42)
Xp = pca.fit_transform(Xz)
evr = pca.explained_variance_ratio_

# Names/markers (edit if you prefer)
label_name = {0: "TD", 1: "CP"}     # or {0:"Non-CP", 1:"CP"}
markers = {0: "o", 1: "^"}          # circle vs triangle

# =========================
# Plot (3D, slide-friendly)
# =========================
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

for lab in sorted(np.unique(y)):
    idx = (y == lab)
    ax.scatter(
        Xp[idx, 0], Xp[idx, 1], Xp[idx, 2],
        s=70, marker=markers.get(lab, "o"),
        alpha=0.90, edgecolors="white", linewidths=0.7,
        label=label_name.get(lab, str(lab))
    )

# Optional: centroids (good for storytelling)
for lab in sorted(np.unique(y)):
    idx = (y == lab)
    c = Xp[idx].mean(axis=0)
    ax.scatter(
        [c[0]], [c[1]], [c[2]],
        s=220, marker="X", linewidths=2.2,
        label=f"{label_name.get(lab, lab)} centroid"
    )

ax.set_title("3D PCA of speech-dynamics features (L-BI) — subject-level", pad=16)
ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}% var.)", labelpad=10)
ax.set_ylabel(f"PC2 ({evr[1]*100:.1f}% var.)", labelpad=10)
ax.set_zlabel(f"PC3 ({evr[2]*100:.1f}% var.)", labelpad=10)

# Clean look for projectors
ax.grid(True, alpha=0.25)
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False

# Nice viewing angle (tweak if you like)
ax.view_init(elev=18, azim=-55)

# Legend outside (prevents covering points)
ax.legend(
    frameon=False,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0.0
)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import PowerNorm, LinearSegmentedColormap
from sklearn.metrics import roc_curve, auc, confusion_matrix

# ============================
# FIGURE SETUP (1x2 PANELS)
# ============================
fig, (ax_cm, ax_roc) = plt.subplots(
    1, 2,
    figsize=(11.5, 4.6),
    dpi=300
)

# ============================
# CONFUSION MATRIX (LEFT)
# ============================
norm = PowerNorm(gamma=0.7, vmin=0, vmax=1)

cm_tot = np.sum(cms, axis=0)
cm_norm = cm_tot / cm_tot.sum(axis=1, keepdims=True)

# Truncated Blues colormap
base = sns.color_palette("Blues", as_cmap=True)
colors = base(np.linspace(0.03, 0.98, 256))
cmap = LinearSegmentedColormap.from_list("Blues_trunc", colors)

sns.heatmap(
    cm_norm,
    annot=cm_tot,
    fmt="d",
    norm=norm,
    cmap=cmap,
    square=True,
    linewidths=0.79,
    linecolor="black",
    cbar=True,
    ax=ax_cm
)

ax_cm.set_title("Confusion Matrix (L-BI)", fontweight="bold", fontsize=16, pad=8)
ax_cm.set_xlabel("Predicted label", fontweight="bold", fontsize=13)
ax_cm.set_ylabel("True label", fontweight="bold", fontsize=13)

ax_cm.set_xticklabels(["TD", "CP"], rotation=0, fontweight="bold", fontsize=12)
ax_cm.set_yticklabels(["TD", "CP"], rotation=90, fontweight="bold", fontsize=12)

for t in ax_cm.texts:
    t.set_fontsize(11)
    t.set_fontweight("bold")

cbar = ax_cm.collections[0].colorbar
cbar.set_ticks([0, 1])
cbar.set_ticklabels(["0%", "100%"])
cbar.ax.tick_params(labelsize=9)

# ============================
# ROC CURVE (RIGHT)
# ============================
C_AGG_ROC  = "#0072B2"
C_MEAN_ROC = "#D55E00"
C_SVM_PT   = "#E69F00"
C_BEST_PT  = "#009E73"
C_CHANCE   = "#7A7A7A"

# Aggregated ROC
fpr_all, tpr_all, _ = roc_curve(y, oof_score, pos_label=1)
auc_all = auc(fpr_all, tpr_all)

ax_roc.plot(
    fpr_all, tpr_all,
    lw=2.8, color=C_AGG_ROC,
    label=f"ROC (Aggregated)  AUC={auc_all:.2f}"
)

# Mean ROC across folds
ax_roc.plot(
    mean_fpr, mean_tpr,
    lw=2.2, linestyle="--", color=C_MEAN_ROC,
    label=f"ROC (Fold mean)  AUC={mean_auc:.2f}±{std_auc:.2f}"
)

# Chance level
ax_roc.plot([0, 1], [0, 1], linestyle=":", lw=1.4, color=C_CHANCE)

# Operating points
ax_roc.scatter(
    fpr_svm, sens_svm,
    s=85, marker="o",
    color=C_SVM_PT,
    edgecolor="black", linewidth=0.7,
    zorder=5,
    label=f"SVM decision (BAcc={bacc_svm:.2f})"
)

ax_roc.scatter(
    fpr_b, sens_b,
    s=95, marker="s",
    color=C_BEST_PT,
    edgecolor="black", linewidth=0.7,
    zorder=6,
    label=f"Best BAcc (BAcc={bacc_b:.2f})"
)

ax_roc.set_xlim(0, 1)
ax_roc.set_ylim(0, 1.05)

ax_roc.set_xlabel("False Positive Rate (1 − Specificity)", fontweight="bold", fontsize=13)
ax_roc.set_ylabel("True Positive Rate (Sensitivity)", fontweight="bold", fontsize=13)
ax_roc.set_title("ROC curve - L-BI", fontweight="bold", fontsize=16)

ax_roc.legend(loc="lower right", frameon=False, fontsize=10)

# ============================
# FINAL LAYOUT
# ============================
plt.tight_layout()
plt.savefig("CM_ROC_LBI_PAPER.png", dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import PowerNorm, LinearSegmentedColormap

# ============================
# PALETTE OBBLIGATORIA
# ============================
NAVY = '#0B1B3B'
TEAL = '#00C2A8'
AMBER = '#FF8A3C'
SLATE = '#5A6B7A'
SLATE_LT = '#B7C0CC'
WHITE = '#FFFFFF'
NAVY_MID = '#12284A'

POSTER_CMAP = LinearSegmentedColormap.from_list(
    'neuro_vocal',
    ['#00C2A8', '#2DCDB7', '#5FE3D0', '#A7E9DB', '#FFB37A', '#FF8A3C']
)

# ============================
# CONFUSION MATRIX (STILE PAPER)
# ============================
norm = PowerNorm(gamma=0.7, vmin=0, vmax=1)

cm_tot = np.sum(cms, axis=0)
cm_norm = cm_tot / cm_tot.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8), dpi=300)
fig.patch.set_facecolor(NAVY)
ax.set_facecolor(NAVY)

hm = sns.heatmap(
    cm_norm,
    annot=cm_tot,
    fmt='d',
    norm=norm,
    cmap=POSTER_CMAP,
    square=True,
    linewidths=0.79,
    linecolor=SLATE,
    cbar=True,
    ax=ax,
    annot_kws={'color': WHITE, 'fontsize': 20, 'fontweight': 'bold'}
)

ax.set_title('Confusion Matrix (L-BI)', color=WHITE, fontsize=24, fontweight='bold', pad=15)
ax.set_xlabel('Predicted label', color=WHITE, fontsize=20)
ax.set_ylabel('True label', color=WHITE, fontsize=20)

ax.set_xticklabels(['TD', 'CP'], rotation=0, fontsize=16, color=SLATE_LT)
ax.set_yticklabels(['TD', 'CP'], rotation=90, fontsize=16, color=SLATE_LT)

ax.tick_params(colors=SLATE_LT, labelsize=16)
ax.spines['bottom'].set_color(SLATE)
ax.spines['left'].set_color(SLATE)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.15, color=SLATE)

cbar = hm.collections[0].colorbar
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['0%', '100%'])
cbar.ax.tick_params(colors=SLATE_LT, labelsize=14)
cbar.outline.set_edgecolor(SLATE)

plt.tight_layout()
plt.savefig('CM_LBI_PAPER_STYLED.png', dpi=300, bbox_inches='tight', facecolor=NAVY, edgecolor='none')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# ============================
# PALETTE OBBLIGATORIA (fallback se non eseguita la cella sopra)
# ============================
if 'NAVY' not in globals():
    NAVY = '#0B1B3B'
    TEAL = '#00C2A8'
    AMBER = '#FF8A3C'
    SLATE = '#5A6B7A'
    SLATE_LT = '#B7C0CC'
    WHITE = '#FFFFFF'
    NAVY_MID = '#12284A'

# ============================
# ROC CURVE (STILE PAPER)
# ============================
fpr_all, tpr_all, _ = roc_curve(y, oof_score, pos_label=1)
auc_all = auc(fpr_all, tpr_all)

fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
fig.patch.set_facecolor(NAVY)
ax.set_facecolor(NAVY)

# Curva principale ROC: AMBER, linewidth=3
ax.plot(
    fpr_all,
    tpr_all,
    lw=3,
    color=AMBER,
    label=f'ROC (Aggregated) AUC={auc_all:.2f}'
)

# Curva media fold mantenuta come seconda serie: TEAL
ax.plot(
    mean_fpr,
    mean_tpr,
    lw=2.5,
    linestyle='-',
    color=TEAL,
    label=f'ROC (Fold mean) AUC={mean_auc:.2f}±{std_auc:.2f}'
)

# Chance line: SLATE, linewidth=1.5, '--'
ax.plot([0, 1], [0, 1], linestyle='--', lw=1.5, color=SLATE)

# Operating points (scatter): cluster 1 TEAL, cluster 2 AMBER
ax.scatter(
    fpr_svm,
    sens_svm,
    s=80,
    alpha=0.8,
    marker='o',
    color=TEAL,
    edgecolor='none',
    zorder=5,
    label=f'SVM decision (BAcc={bacc_svm:.2f})'
)

ax.scatter(
    fpr_b,
    sens_b,
    s=80,
    alpha=0.8,
    marker='s',
    color=AMBER,
    edgecolor='none',
    zorder=6,
    label=f'Best BAcc (BAcc={bacc_b:.2f})'
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)

ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.xaxis.label.set_color(WHITE)
ax.xaxis.label.set_fontsize(20)
ax.yaxis.label.set_color(WHITE)
ax.yaxis.label.set_fontsize(20)

ax.set_title('ROC curve - L-BI', color=WHITE, fontsize=24, fontweight='bold', pad=15)

ax.tick_params(colors=SLATE_LT, labelsize=16)
ax.spines['bottom'].set_color(SLATE)
ax.spines['left'].set_color(SLATE)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend = ax.legend(
    fontsize=14,
    facecolor=NAVY_MID,
    edgecolor=SLATE,
    labelcolor=WHITE,
    framealpha=0.9,
    loc='lower right'
)

ax.grid(True, alpha=0.15, color=SLATE)

plt.tight_layout()
plt.savefig('ROC_LBI_PAPER_STYLED.png', dpi=300, bbox_inches='tight', facecolor=NAVY, edgecolor='none')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import PowerNorm, LinearSegmentedColormap
from matplotlib import patheffects as pe
from sklearn.metrics import roc_curve, auc
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ============================
# PALETTE OBBLIGATORIA
# ============================
NAVY = '#0B1B3B'
TEAL = '#00C2A8'
AMBER = '#FF8A3C'
SLATE = '#5A6B7A'
SLATE_LT = '#B7C0CC'
WHITE = '#FFFFFF'
NAVY_MID = '#12284A'

POSTER_CMAP = LinearSegmentedColormap.from_list(
    'neuro_vocal',
    ['#00C2A8', '#2DCDB7', '#5FE3D0', '#A7E9DB', '#FFB37A', '#FF8A3C']
)

# ============================
# FIGURE SETUP (1x2 LAYOUT)
# ============================
fig, (ax_cm, ax_roc) = plt.subplots(
    1, 2,
    figsize=(13, 6),
    dpi=300
)
fig.patch.set_facecolor(NAVY)

# ============================
# CONFUSION MATRIX (LEFT)
# ============================
ax_cm.set_facecolor(NAVY)
norm = PowerNorm(gamma=0.7, vmin=0, vmax=1)

cm_tot = np.sum(cms, axis=0)
cm_norm = cm_tot / cm_tot.sum(axis=1, keepdims=True)

hm = sns.heatmap(
    cm_norm,
    annot=cm_tot,
    fmt='d',
    norm=norm,
    cmap=POSTER_CMAP,
    square=True,
    linewidths=0.79,
    linecolor=SLATE,
    cbar=False,  # IMPORTANTE: no colorbar automatica
    ax=ax_cm,
    annot_kws={'color': NAVY, 'fontsize': 20, 'fontweight': 'bold'}
)

# Colorbar controllata dentro il subplot
divider = make_axes_locatable(ax_cm)
cax = divider.append_axes("right", size="5%", pad="2%")
cbar = fig.colorbar(hm.collections[0], cax=cax)
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['0%', '100%'])
cbar.ax.tick_params(colors=SLATE_LT, labelsize=12)
cbar.outline.set_edgecolor(SLATE)

'''
    annot_kws={
        'color': WHITE, 
        'fontsize': 18, 
        'fontweight': 'bold',
        'path_effects': [pe.withStroke(linewidth=1.5, foreground='black')]
    }
'''
ax_cm.set_title('Confusion Matrix (L-BI)', color=WHITE, fontsize=20, fontweight='bold', pad=12)
ax_cm.set_xlabel('Predicted label', color=WHITE, fontsize=18, fontweight='bold')
ax_cm.set_ylabel('True label', color=WHITE, fontsize=18, fontweight='bold')

ax_cm.set_xticklabels(['TD', 'CP'], rotation=0, fontsize=14, color=SLATE_LT)
ax_cm.set_yticklabels(['TD', 'CP'], rotation=90, fontsize=14, color=SLATE_LT)

ax_cm.tick_params(colors=SLATE_LT, labelsize=14)
ax_cm.spines['bottom'].set_color(SLATE)
ax_cm.spines['left'].set_color(SLATE)
ax_cm.spines['top'].set_visible(False)
ax_cm.spines['right'].set_visible(False)
# ax_cm.grid(True, alpha=0.15, color=SLATE)

cbar = hm.collections[0].colorbar
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['0%', '100%'])
cbar.ax.tick_params(colors=SLATE_LT, labelsize=12)
cbar.outline.set_edgecolor(SLATE)

# ============================
# ROC CURVE (RIGHT)
# ============================
ax_roc.set_facecolor(NAVY_MID)

fpr_all, tpr_all, _ = roc_curve(y, oof_score, pos_label=1)
auc_all = auc(fpr_all, tpr_all)

# Curva principale ROC: AMBER, linewidth=3
ax_roc.plot(
    fpr_all, tpr_all,
    lw=3, color=AMBER,
    label=f'ROC (Agg) AUC={auc_all:.2f}'
)

# Curva media fold: TEAL
ax_roc.plot(
    mean_fpr, mean_tpr,
    lw=2.5, linestyle='-', color=TEAL,
    label=f'Mean AUC={mean_auc:.2f}±{std_auc:.2f}'
)

# Chance line: SLATE
ax_roc.plot([0, 1], [0, 1], linestyle='--', lw=1.5, color=SLATE)

# Operating points
ax_roc.scatter(
    fpr_svm, sens_svm,
    s=70, alpha=0.8, marker='o', color=TEAL,
    edgecolor='none', zorder=5,
    label=f'SVM decision (BAcc={bacc_svm:.2f})'
)

ax_roc.scatter(
    fpr_b, sens_b,
    s=70, alpha=0.8, marker='s', color=AMBER,
    edgecolor='none', zorder=6,
    label=f'Best BAcc (BAcc={bacc_b:.2f})'
)

ax_roc.set_xlim(0, 1)
ax_roc.set_ylim(0, 1.05)

ax_roc.set_xlabel('False Positive Rate (1 - Specificity)', color=WHITE, fontsize=18, fontweight='bold')
ax_roc.set_ylabel('True Positive Rate (Sensitivity)', color=WHITE, fontsize=18, fontweight='bold')
ax_roc.set_title('ROC curve - L-BI', color=WHITE, fontsize=20, fontweight='bold', pad=12)

ax_roc.tick_params(colors=SLATE_LT, labelsize=14)
ax_roc.spines['bottom'].set_color(SLATE)
ax_roc.spines['left'].set_color(SLATE)
ax_roc.spines['top'].set_visible(False)
ax_roc.spines['right'].set_visible(False)

legend = ax_roc.legend(
    fontsize=11, facecolor=NAVY_MID, edgecolor=SLATE,
    labelcolor=WHITE, framealpha=0.9, loc='lower right'
)

ax_roc.grid(True, alpha=0.15, color=SLATE)

# ============================
# SAVE & SHOW
# ============================
plt.tight_layout()
plt.savefig('CM_ROC_LBI_PAPER_STYLED.png', dpi=300, bbox_inches='tight', facecolor=NAVY, edgecolor='none')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import PowerNorm, LinearSegmentedColormap
from sklearn.metrics import roc_curve, auc
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ============================
# PALETTE OBBLIGATORIA
# ============================
NAVY = '#0B1B3B'
TEAL = '#00C2A8'
AMBER = '#FF8A3C'
SLATE = '#5A6B7A'
SLATE_LT = '#B7C0CC'
WHITE = '#FFFFFF'
NAVY_MID = '#E9ECEF'

POSTER_CMAP = LinearSegmentedColormap.from_list(
'neuro_vocal',
['#00C2A8', '#2DCDB7', '#5FE3D0', '#A7E9DB', '#FFB37A', '#FF8A3C']
)

# ============================
# FIGURE SETUP (1x2 LAYOUT)
# ============================
fig, (ax_cm, ax_roc) = plt.subplots(
1, 2,
figsize=(13, 6),
dpi=300
)
fig.patch.set_facecolor(NAVY_MID)

# ============================
# CONFUSION MATRIX (LEFT)
# ============================
ax_cm.set_facecolor(NAVY_MID)
norm = PowerNorm(gamma=0.7, vmin=0, vmax=1)

cm_tot = np.sum(cms, axis=0)
cm_norm = cm_tot / cm_tot.sum(axis=1, keepdims=True)

hm = sns.heatmap(
cm_norm,
annot=cm_tot,
fmt='d',
norm=norm,
cmap=POSTER_CMAP,
square=True,
linewidths=0.79,
linecolor=SLATE,
cbar=False,
ax=ax_cm,
annot_kws={'color': NAVY, 'fontsize': 20, 'fontweight': 'bold'}
)

# Colorbar controllata dentro il subplot
divider = make_axes_locatable(ax_cm)
cax = divider.append_axes("right", size="5%", pad="2%")
cbar = fig.colorbar(hm.collections[0], cax=cax)
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['0%', '100%'])
cbar.ax.tick_params(colors=NAVY, labelsize=12)
cbar.outline.set_edgecolor(NAVY)

ax_cm.set_title('Confusion Matrix (L-BI)', color=NAVY, fontsize=20, fontweight='bold', pad=12)
ax_cm.set_xlabel('Predicted label', color=NAVY, fontsize=18, fontweight='bold')
ax_cm.set_ylabel('True label', color=NAVY, fontsize=18, fontweight='bold')

ax_cm.set_xticklabels(['TD', 'CP'], rotation=0, fontsize=14, color=NAVY)
ax_cm.set_yticklabels(['TD', 'CP'], rotation=90, fontsize=14, color=NAVY)

ax_cm.tick_params(colors=NAVY, labelsize=14)
ax_cm.spines['bottom'].set_color(NAVY)
ax_cm.spines['left'].set_color(NAVY)
ax_cm.spines['top'].set_visible(False)
ax_cm.spines['right'].set_visible(False)

# ============================
# ROC CURVE (RIGHT)
# ============================
ax_roc.set_facecolor(NAVY_MID)

fpr_all, tpr_all, _ = roc_curve(y, oof_score, pos_label=1)
auc_all = auc(fpr_all, tpr_all)

# Curva principale ROC: AMBER
ax_roc.plot(
fpr_all, tpr_all,
lw=3,
color=AMBER,
label=f'ROC (Agg) AUC={auc_all:.2f}'
)

# Curva media fold: TEAL
ax_roc.plot(
mean_fpr, mean_tpr,
lw=2.5,
linestyle='-',
color=TEAL,
label=f'Mean AUC={mean_auc:.2f}±{std_auc:.2f}'
)

# Chance line: SLATE
ax_roc.plot([0, 1], [0, 1], linestyle='--', lw=1.5, color=NAVY)

# Operating points
ax_roc.scatter(
fpr_svm, sens_svm,
s=70, alpha=0.8, marker='o',
color=TEAL, edgecolor='none', zorder=5,
label=f'SVM decision (BAcc={bacc_svm:.2f})'
)

ax_roc.scatter(
fpr_b, sens_b,
s=70, alpha=0.8, marker='s',
color=AMBER, edgecolor='none', zorder=6,
label=f'Best BAcc (BAcc={bacc_b:.2f})'
)

ax_roc.set_xlim(0, 1)
ax_roc.set_ylim(0, 1.05)

ax_roc.set_xlabel('False Positive Rate (1 - Specificity)', color=NAVY, fontsize=18, fontweight='bold')
ax_roc.set_ylabel('True Positive Rate (Sensitivity)', color=NAVY, fontsize=18, fontweight='bold')
ax_roc.set_title('ROC curve - L-BI', color=NAVY, fontsize=20, fontweight='bold', pad=12)

ax_roc.tick_params(colors=NAVY, labelsize=14)
ax_roc.spines['bottom'].set_color(NAVY)
ax_roc.spines['left'].set_color(NAVY)
ax_roc.spines['top'].set_visible(False)
ax_roc.spines['right'].set_visible(False)

legend = ax_roc.legend(
fontsize=11,
facecolor=NAVY_MID,
edgecolor=NAVY,
labelcolor=NAVY,
framealpha=0.9,
loc='lower right'
)

ax_roc.grid(True, alpha=0.15, color=NAVY)

# ============================
# SAVE & SHOW
# ============================
plt.tight_layout()
plt.savefig('CM_ROC_LBI_PAPER_STYLED.png', dpi=300, bbox_inches='tight', facecolor=NAVY_MID, edgecolor='none')
plt.show()